# Submit a Command Job

Run a simple taxi-data summarization script on the configured Azure ML compute and inspect its terminal status and output URI.

**Sources:** Adapted from this repository's command-job patterns and the [Azure ML single-step examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/jobs/single-step), MIT License.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
EXPERIMENT_NAME = os.environ["WORKSHOP_EXPERIMENT_NAME"]
SUBMIT = os.getenv("SUBMIT_FOUNDATION_JOB", "false").lower() in {"1", "true", "yes"}

In [ ]:
job = command(
    display_name="Workshop taxi CSV summary",
    experiment_name=EXPERIMENT_NAME,
    code=str(WORKSHOP_ROOT / "src/foundations/command_job"),
    command=(
        "python summarize.py "
        "--input-data ${{inputs.input_data}} "
        "--output-data ${{outputs.output_data}}"
    ),
    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=str(WORKSHOP_ROOT / "data/taxi/raw/yellowTaxiData.csv"),
        )
    },
    outputs={
        "output_data": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount")
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute=f"azureml:{COMPUTE_NAME}",
    tags={"workshop": "azureml-h2o", "operation": "command-job"},
)

if SUBMIT:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted job: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Job ended with status {final_job.status}")
    print(f"Output URI: {final_job.outputs['output_data'].path}")
else:
    print(f"Prepared command job for compute: {COMPUTE_NAME}")
    print("Submission disabled. Set SUBMIT_FOUNDATION_JOB=true in workshop/.env.")

## Expected Result

The command job completes on the configured cluster and publishes a `summary.json` artifact in its output folder.

Next: `06_deploy_online_endpoint.ipynb`.